# Wisconsin Fast Forward Grant — Training & Development Contact Enrichment

This notebook automates finding and populating **Training & Development / HR decision-makers** (up to 2 contacts per organization) for Wisconsin Fast Forward grant recipients.

### How It Works
1. **Load & Standardize:** Reads the grant recipient list and InterPro's historical participant export, normalizing company names (removing `LLC`, `Inc.`, division notes, etc.).
2. **Internal Cross-Reference:** Fuzzy-matches grant recipients against past InterPro attendees to pull existing T&D/HR contacts and identify verified corporate email domains/naming patterns (e.g., `first.last@company.com`).
3. **Automated Web Search & Enrichment:** Uses the Serper Google Search API to locate Wisconsin-based Training, L&D, or HR leaders on LinkedIn for remaining companies, infers their work email using the company's domain pattern, and writes all fields directly into Columns **W–AF** of a new Excel file (`WisTRAIN_Grant_Results_Enriched.xlsx`).

---

### Setup & API Key Instructions (Required Before Running)
To keep API credentials secure on GitHub, this notebook reads your search key from an environment variable (`SERPER_API_KEY`) rather than hardcoding it.

1. **Get a Free API Key:** Sign up at [serper.dev](https://serper.dev) (includes 2,500 free search queries; no credit card required) and copy your API key from the dashboard.
2. **Install Required Packages:**
   ```bash
   pip install pandas openpyxl rapidfuzz requests python-dotenv

In [1]:
import pandas as pd
import re
from rapidfuzz import process, fuzz

# 1. Define the file names for the Wisconsin Fast Forward grant sheet and InterPro's historical participant export
grants_path = "WisTRAIN Grant Results (2026) copy.xlsx"
participants_path = "ParticipantDataReport 11-18-2026 copy.xlsx"

# 2. Load both Excel files into tables (header=3 skips the top title banner so Row 4 becomes the column headers)
grants_df = pd.read_excel(grants_path, sheet_name="Grant Recipients", header=3)
participants_df = pd.read_excel(participants_path)

print(f"Loaded {len(grants_df)} grant recipient companies and {len(participants_df)} historical InterPro participant records.")

Loaded 85 grant recipient companies and 164639 historical InterPro participant records.


In [2]:
# Helper function to standardize company names so minor formatting differences don't prevent matches
# (e.g., "Matenaer Corporation" and "Matenaer" both become "matenaer")
def clean_company(name):
    if pd.isna(name):
        return ""
    name = str(name).lower()
    # Remove parenthetical notes such as "(Wisconsin Centrifugal Division)"
    name = re.sub(r"\(.*?\)", "", name)
    # Remove common business suffixes (LLC, Inc., Corp, etc.) and punctuation
    name = re.sub(r"\b(llc|inc|corp|corporation|co|ltd|company|group)\b\.?", "", name)
    name = re.sub(r"[^a-z0-9\s]", "", name)
    return " ".join(name.split())

# Apply the cleanup function to both spreadsheets
grants_df["clean_org"] = grants_df["Company Name"].apply(clean_company)
participants_df["clean_org"] = participants_df["Organization Name"].apply(clean_company)

# Keep only InterPro participant records that have both a valid company name and an email address
valid_parts = participants_df[
    (participants_df["clean_org"] != "") & (participants_df["Email Address"].notna())
].copy()

In [3]:
from collections import Counter

# Keywords used to identify existing InterPro attendees who work in Training, Learning & Development, or HR
td_keywords = r"training|learning|human resources|\bhr\b|talent|people|(?:learning|training|talent|organizational|workforce)\s+(?:&|and)?\s*development"

# Ignore personal email providers so we only capture official corporate domains
excluded_domains = {"gmail.com", "yahoo.com", "hotmail.com", "outlook.com", "aol.com", "icloud.com", "usps.gov"}

unique_part_orgs = valid_parts["clean_org"].unique().tolist()
enriched_rows = []

# Loop through every company on the Wisconsin Fast Forward grant list
for idx, row in grants_df.iterrows():
    grant_org = row["clean_org"]
    result = {
        "Grant Company": row["Company Name"],
        "Matched InterPro Org": "",
        "Match Score": 0,
        "Corporate Domain": "",
        "Inferred Email Pattern": "",
        "Total Past Attendees": 0,
        "C1_First": "", "C1_Last": "", "C1_Title": "", "C1_Email": "", "C1_LinkedIn": "",
        "C2_First": "", "C2_Last": "", "C2_Title": "", "C2_Email": "", "C2_LinkedIn": "",
    }
    
    if grant_org:
        # Compare the grant company name against all organizations in InterPro's history (92%+ similarity required)
        best_match = process.extractOne(grant_org, unique_part_orgs, scorer=fuzz.token_sort_ratio)
        
        if best_match and best_match[1] >= 92:
            matched_clean_org = best_match[0]
            org_people = valid_parts[valid_parts["clean_org"] == matched_clean_org]
            
            # Identify the company's primary work email domain (e.g., "metaltek.com") from past participants
            domains = [
                e.split("@")[-1].lower() for e in org_people["Email Address"].dropna()
                if "@" in str(e) and e.split("@")[-1].lower() not in excluded_domains
            ]
            top_domain = Counter(domains).most_common(1)[0][0] if domains else ""
            
            # Save one sample corporate email so we know the company's naming format (e.g., first.last@company.com)
            corp_emails = [
                e for e in org_people["Email Address"].dropna()
                if top_domain and str(e).lower().endswith(f"@{top_domain}")
            ]
            sample_email = corp_emails[0] if corp_emails else ""
            
            # Check if any past InterPro attendees at this company already hold T&D or HR titles
            td_people = org_people[
                org_people["Title"].fillna("").str.contains(td_keywords, case=False, regex=True)
            ].drop_duplicates(subset=["Email Address"])
            
            result.update({
                "Matched InterPro Org": org_people["Organization Name"].iloc[0],
                "Match Score": round(best_match[1], 1),
                "Corporate Domain": top_domain,
                "Inferred Email Pattern": sample_email,
                "Total Past Attendees": len(org_people),
            })
            
            # If we already have T&D/HR contacts in InterPro's records, fill in Contact 1 (and Contact 2 if available)
            if len(td_people) >= 1:
                result["C1_First"] = td_people["First Name"].iloc[0]
                result["C1_Last"] = td_people["Last Name"].iloc[0]
                result["C1_Title"] = td_people["Title"].iloc[0]
                result["C1_Email"] = td_people["Email Address"].iloc[0]
            if len(td_people) >= 2:
                result["C2_First"] = td_people["First Name"].iloc[1]
                result["C2_Last"] = td_people["Last Name"].iloc[1]
                result["C2_Title"] = td_people["Title"].iloc[1]
                result["C2_Email"] = td_people["Email Address"].iloc[1]
                
    enriched_rows.append(result)

# Store the intermediate results in a table
enriched_df = pd.DataFrame(enriched_rows)
print(f"Found {len(enriched_df[enriched_df['Corporate Domain'] != ''])} companies with existing InterPro history and email domains.")

Found 23 companies with existing InterPro history and email domains.


In [ ]:
import openpyxl
import requests
import time
import re
from urllib.parse import urlparse

# API key for automated Google search queries (via Serper.dev)
SERPER_API_KEY = "PASTE_YOUR_SERPER_KEY_HERE"

# Filter to ensure we only keep Training, HR, or Leadership decision-makers (excludes unrelated roles)
valid_role_re = re.compile(
    r"training|learning|human resources|\bhr\b|talent|people|organizational|workforce|director|manager|vp|vice president|president|ceo|owner|operations",
    re.IGNORECASE
)

# Clean up a person's name by removing certifications/credentials (e.g., "Jamie Aggen, PHR" -> "Jamie", "Aggen")
def clean_person_name(raw_name):
    name = raw_name.split(",")[0].strip()
    name = re.sub(r"\(.*?\)", "", name).strip()
    tokens = name.split()
    first = tokens[0] if len(tokens) >= 1 else ""
    last = tokens[-1] if len(tokens) >= 2 else ""
    return first, last

# Helper function to run a targeted Google search and return the top organic results
def serper_search(query, num=5):
    url = "https://google.serper.dev/search"
    headers = {"X-API-KEY": SERPER_API_KEY, "Content-Type": "application/json"}
    resp = requests.post(url, headers=headers, json={"q": query, "num": num}, timeout=15)
    return resp.json().get("organic", []) if resp.status_code == 200 else []

# Find a company's official website domain via web search if it wasn't in InterPro's participant history
def get_company_domain(company_name):
    clean_name = re.sub(r"\(.*?\)", "", str(company_name)).strip()
    items = serper_search(f'"{clean_name}" Wisconsin', num=3)
    skip_hosts = ("linkedin.com", "facebook.com", "dnb.com", "bloomberg.com", "zoominfo.com", "mapquest.com", "yelp.com", "wisconsin.gov")
    for item in items:
        host = urlparse(item.get("link", "")).netloc.lower().replace("www.", "")
        if host and not any(s in host for s in skip_hosts):
            return host
    return ""

# Search LinkedIn profiles for Wisconsin-based Training, L&D, or HR contacts at the target company
def search_linkedin_td(company_name):
    clean_name = re.sub(r"\(.*?\)", "", str(company_name)).strip()
    query = f'site:linkedin.com/in "{clean_name}" "Wisconsin" ("Training" OR "Learning and Development" OR "Human Resources" OR "HR Manager" OR "Talent")'
    items = serper_search(query, num=5)
    
    results = []
    for item in items:
        link = item.get("link", "")
        title_str = item.get("title", "")
        snippet = item.get("snippet", "")
        if "linkedin.com/in/" not in link:
            continue
            
        # LinkedIn search titles are typically structured as: "First Last - Job Title - Company | LinkedIn"
        parts = [p.strip() for p in title_str.replace("| LinkedIn", "").split(" - ")]
        first, last = clean_person_name(parts[0]) if len(parts) >= 1 else ("", "")
        role = parts[1] if len(parts) >= 2 else ""
        
        # If LinkedIn displays the city/state instead of job title, extract the role from the profile snippet
        if "wisconsin" in role.lower() or "united states" in role.lower():
            role = snippet.split("·")[0].strip()[:80]
            
        # Keep up to 2 contacts whose title/bio matches our HR, Training, or Leadership keywords
        combined_text = f"{role} {snippet}"
        if first and last and valid_role_re.search(combined_text):
            results.append({"first": first, "last": last, "title": role, "linkedin": link})
            
    return results[:2]

# Generate a likely work email using the company's domain and naming convention (e.g., first.last@company.com)
def infer_email(first, last, domain, sample_email=""):
    if not first or not last or not domain:
        return ""
    f = re.sub(r"[^a-z]", "", first.lower())
    l = re.sub(r"[^a-z]", "", last.lower())
    if not f or not l:
        return ""
    
    local_sample = sample_email.split("@")[0].lower() if "@" in str(sample_email) else ""
    if "." in local_sample:
        return f"{f}.{l}@{domain}"
    elif len(local_sample) > 1 and local_sample.startswith(f[0]) and not local_sample.startswith(f):
        if local_sample.endswith(l):
            return f"{f[0]}{l}@{domain}"   # Pattern: flast@company.com
        elif local_sample.startswith(f):
            return f"{f}{l[0]}@{domain}"   # Pattern: firstl@company.com
    return f"{f}.{l}@{domain}"             # Default pattern: first.last@company.com

# Step through every grant company that does not already have a contact and search the web
for idx, row in enriched_df.iterrows():
    if row["C1_First"] != "":
        continue
        
    comp = row["Grant Company"]
    domain = row["Corporate Domain"]
    sample = row["Inferred Email Pattern"]
    
    found = search_linkedin_td(comp)
    if found:
        # Look up the company's website domain if we didn't already get it from InterPro's participant list
        if not domain:
            domain = get_company_domain(comp)
            enriched_df.at[idx, "Corporate Domain"] = domain
            
        # Populate Contact 1 fields
        enriched_df.at[idx, "C1_First"] = found[0]["first"]
        enriched_df.at[idx, "C1_Last"] = found[0]["last"]
        enriched_df.at[idx, "C1_Title"] = found[0]["title"]
        enriched_df.at[idx, "C1_LinkedIn"] = found[0]["linkedin"]
        enriched_df.at[idx, "C1_Email"] = infer_email(found[0]["first"], found[0]["last"], domain, sample)
        
        # Populate Contact 2 fields if a second matching contact was found
        if len(found) >= 2:
            enriched_df.at[idx, "C2_First"] = found[1]["first"]
            enriched_df.at[idx, "C2_Last"] = found[1]["last"]
            enriched_df.at[idx, "C2_Title"] = found[1]["title"]
            enriched_df.at[idx, "C2_LinkedIn"] = found[1]["linkedin"]
            enriched_df.at[idx, "C2_Email"] = infer_email(found[1]["first"], found[1]["last"], domain, sample)
            
    time.sleep(0.3)  # Brief pause between requests to respect API rate limits

# Open the original formatted grant spreadsheet and write our contacts directly into Columns W through AF
wb = openpyxl.load_workbook(grants_path)
ws = wb["Grant Recipients"]

# Row 5 in Excel corresponds to the first company row (index 0)
for idx, row in enriched_df.iterrows():
    excel_row = idx + 5
    ws[f"W{excel_row}"] = row["C1_First"]
    ws[f"X{excel_row}"] = row["C1_Last"]
    ws[f"Y{excel_row}"] = row["C1_Title"]
    ws[f"Z{excel_row}"] = row["C1_Email"]
    ws[f"AA{excel_row}"] = row["C1_LinkedIn"]
    ws[f"AB{excel_row}"] = row["C2_First"]
    ws[f"AC{excel_row}"] = row["C2_Last"]
    ws[f"AD{excel_row}"] = row["C2_Title"]
    ws[f"AE{excel_row}"] = row["C2_Email"]
    ws[f"AF{excel_row}"] = row["C2_LinkedIn"]

# Save the updated spreadsheet as a new file so the original remains untouched
output_file = "WisTRAIN_Grant_Results_Enriched.xlsx"
wb.save(output_file)

filled_count = (enriched_df["C1_First"] != "").sum()
print(f"Done! Populated contacts for {filled_count} / {len(enriched_df)} companies and saved to {output_file}")

/Users/god./Downloads/interpro_grant_contacts/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
